# Reference · Day 2 studio — the smallest honest pipeline

**Not a marking key.** There is no single right answer to "build the smallest honest
pipeline you can defend" — you chose your own columns, and so did everyone else.

This is here so you can compare. Run it, read it, and check your own notebook against
the five questions further down. If yours differs, the interesting question is *why*, and that
is a better conversation than "was I right".

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")

import numpy as np
import pandas as pd
from stat764 import load

ames = load("ames.csv")

## One reference implementation

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric     = ["Gr_Liv_Area", "Year_Built", "Overall_Qual", "Total_Bsmt_SF"]
categorical = ["Neighborhood", "Central_Air", "Garage_Type"]

X_train, X_test, y_train, y_test = train_test_split(
    ames[numeric + categorical], ames["SalePrice"], test_size=0.25, random_state=764)

pipe = Pipeline([
    ("prep", ColumnTransformer([
        ("num", Pipeline([("fill",   SimpleImputer(strategy="median")),
                          ("scale",  StandardScaler())]),                      numeric),
        ("cat", Pipeline([("fill",   SimpleImputer(strategy="constant",
                                                   fill_value="Missing")),
                          ("encode", OneHotEncoder(handle_unknown="ignore"))]), categorical),
    ])),
    ("model", LinearRegression()),
])

pipe.fit(X_train, y_train)

baseline = np.full(len(y_test), y_train.mean())
print(f"  mean baseline        R-squared {r2_score(y_test, baseline):.3f}")
print(f"  this pipeline        R-squared {r2_score(y_test, pipe.predict(X_test)):.3f}")
print(f"  columns in {len(numeric) + len(categorical)}, "
      f"after encoding {pipe.named_steps['prep'].transform(X_train).shape[1]}")

## Your answer probably differs, and that is fine

Here is the range the same skeleton produces with different column choices. Find where
yours sits.

In [ ]:
allnum = [c for c in ames.select_dtypes("number").columns if c != "SalePrice"]
# pandas 3 gave strings their own dtype, so select_dtypes(include="object")
# now warns that it will stop matching them. "Not numeric" is what we actually
# mean, and it says the same thing on every pandas version.
allcat = [c for c in ames.columns if not pd.api.types.is_numeric_dtype(ames[c])]

def try_columns(num, cat):
    X = ames[num + cat]
    Xtr, Xte, ytr, yte = train_test_split(
        X, ames["SalePrice"], test_size=0.25, random_state=764)
    steps = []
    if num:
        steps.append(("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                                       ("scale", StandardScaler())]), num))
    if cat:
        steps.append(("cat", Pipeline([("fill", SimpleImputer(strategy="constant",
                                                              fill_value="Missing")),
                                       ("encode", OneHotEncoder(handle_unknown="ignore"))]), cat))
    p = Pipeline([("prep", ColumnTransformer(steps)),
                  ("model", LinearRegression())]).fit(Xtr, ytr)
    return r2_score(yte, p.predict(Xte)), p.named_steps["prep"].transform(Xtr[:5]).shape[1]

print(f"  {'columns chosen':<34}{'in':>4}{'encoded':>9}{'R2':>8}")
for label, num, cat in [
    ("one predictor",            ["Gr_Liv_Area"], []),
    ("three numeric",            ["Gr_Liv_Area", "Overall_Qual", "Year_Built"], []),
    ("the reference above",      numeric, categorical),
    ("every numeric column",     allnum, []),
    ("everything, as the dtypes fall", allnum, allcat),
]:
    r, n = try_columns(num, cat)
    print(f"  {label:<34}{len(num)+len(cat):>4}{n:>9}{r:>8.3f}")

In [ ]:
# ⚠ Before you read that last row as "everything, handled properly" -- it is not.
coded_looking = [c for c in allnum if ames[c].nunique() <= 16]
print(f"  allcat found {len(allcat)} columns, and every one of them is stored as TEXT.")
print(f"  It found NO integer columns, because it cannot: it sorts on dtype.\n")
print(f"  So these {len(coded_looking)} sit in allnum and were STANDARDISED as if they were")
print( "  quantities, which several of them are not:")
print( "    " + ", ".join(coded_looking))
print(f"\n  MS_SubClass is a dwelling-type CODE — 20 and 60 are labels, not amounts.")
print( "  Mo_Sold is a month. Scaling either one is meaningless.")

## ⚠ `allcat` does not mean what it looks like it means

It means **"stored as text"**, not **"is a categorical variable."** No `dtype` rule can
tell the difference, because the difference is not in the file — it is in what the
numbers *stand for*.

So the row labelled *everything, as the dtypes fall* is exactly that: every column
included, sorted by how pandas happens to store it. `MS_SubClass`, `Mo_Sold` and a dozen
others went through the numeric branch and got standardised. The model then treats
dwelling type 60 as three times dwelling type 20.

It still scores well, which is the trap. **A number going up is not evidence that the
columns were handled correctly.**

**The fix, in two steps:**

1. **Read the data dictionary and write the list down.** Sort the low-cardinality
   integers into genuine counts (`Fireplaces`), ordered ratings (`Overall_Qual` — your
   call either way), and codes (`MS_SubClass`, `Mo_Sold`). This is a judgment, not a
   lookup, and it is the part nobody can automate for you.
2. **Cast the codes to text before routing them**, so they reach the encoder instead of
   the scaler:

```python
CODES = ["MS_SubClass", "Mo_Sold"]
fixed = ames.copy()
for c in CODES:
    fixed[c] = fixed[c].astype(str)          # or the encoder cannot impute them
numeric     = [c for c in allnum if c not in CODES]
categorical = allcat + CODES
```

The appendix at the bottom does this properly, with two helpers you can copy — and
measures whether it changes the answer. (Briefly: it barely does. Do it anyway, and
the appendix says why that is the argument *for* it rather than against.)

Notice the shape of that: one good predictor gets you two thirds of the way, and the
next thirty-odd columns buy a fraction of what the first three did. Notice too that
*every numeric column* does worse than a smaller set that includes some categoricals.

More is not reliably better — which is the thread Meeting 3 picks up.

## Check your own notebook against this

Not "is my number as high as theirs". These:

| | |
|---|---|
| **1** | Did you call `train_test_split` **before** anything else touched the data? Scroll up and check — this is the one people get wrong while believing they did not. |
| **2** | Is every preprocessing step **inside** the `Pipeline`? If you have a `.fit_transform(` anywhere outside it, that is a leak. |
| **3** | Did you compute a **baseline** and beat it? If you never computed one, you do not actually know whether your model is doing anything. |
| **4** | Would you **put your number on the board** and defend where it came from? |
| **5** | Does the notebook run top to bottom after **Restart & Run All**? |

If any of 1, 2 or 5 is a no, fix it before Lab 2 — those three are the habits the rest
of the course is built on. If 3 or 4 is a no, that is worth two minutes of conversation
rather than a fix.

⚠ **A smaller model that passes all five is a better answer than a larger one that
fails any of them.** "Smallest you can defend" was the brief, and defensible beats big.

---

# Appendix — choosing columns

Everything above used seven columns I picked by hand. Two questions come straight out
of that, and neither was obvious on Thursday.

## A. Which columns are legitimately numeric?

`dtype` will not tell you. Ames has three kinds of integer column and only one of them
is a quantity:

| pile | example | what it is |
|---|---|---|
| **Identifier** | `Order`, `PID` | a serial number. Not a fact about the house. **Drop it.** |
| **Coded category** | `MS_SubClass`, `Mo_Sold` | an integer standing for a label. Scaling it is nonsense. |
| **Genuine quantity** | `Gr_Liv_Area`, `Year_Built` | a number that means a number. |

Ordered ratings like `Overall_Qual` sit between the last two and are defensible either
way. That is a modelling decision, not a lookup.

Two cheap checks find most of the trouble.

In [ ]:
ids = [c for c in ames.columns if ames[c].is_unique]
print(f"unique for every row, so an identifier: {ids}")
print(f"  PID correlates {ames.PID.corr(ames.SalePrice):+.3f} with price — its first three")
print(f"  digits encode the neighbourhood, so it is geography in disguise.\n")

low_card = [c for c in ames.select_dtypes("number").columns
            if c != "SalePrice" and ames[c].nunique() <= 16]
print(f"numeric but with 16 or fewer distinct values ({len(low_card)}):")
print("   ", ", ".join(low_card))
print("\n  Sort those yourself. Fireplaces is a count; Mo_Sold is a month; MS_SubClass is")
print("  a dwelling-type code. Only you can tell, and the data dictionary is the way.")

## B. All numeric, all categorical, or everything

Three one-liners. `select_dtypes` picks columns by type, and `"number"` covers int and
float together.

In [ ]:
all_numeric     = [c for c in ames.select_dtypes("number").columns if c != "SalePrice"]
all_categorical = [c for c in ames.columns
                   if not pd.api.types.is_numeric_dtype(ames[c])]

print(f"  all_numeric      {len(all_numeric):>3} columns")
print(f"  all_categorical  {len(all_categorical):>3} columns")
print(f"  everything       {len(all_numeric) + len(all_categorical):>3} columns")

# ⚠ The one thing to understand about that split.
missed = [c for c in all_numeric if c in low_card or c in ids]
print(f"\n  ⚠ all_categorical means STORED AS TEXT, not IS A CATEGORY.")
print(f"    It contains no integer columns at all: "
      f"{any(pd.api.types.is_integer_dtype(ames[c]) for c in all_categorical)}")
print(f"    So these {len(missed)} land in all_numeric and get STANDARDISED:")
print("      " + ", ".join(missed))

Inside a pipeline there is a version that selects **at fit time** rather than now, which
is what you want if the incoming columns might change:

```python
from sklearn.compose import make_column_selector

ColumnTransformer([
    ("num", numeric_steps,     make_column_selector(dtype_include="number")),
    ("cat", categorical_steps, make_column_selector(dtype_exclude="number")),
])
```

⚠ Convenient, and it makes exactly the mistake section A is about: it files
`MS_SubClass` as numeric, because it is an integer.

**There is no one-liner for "all the categorical variables."** Every shortcut here —
`select_dtypes`, `make_column_selector`, the not-numeric test above — sorts by how a
column is *stored*, and the thing you care about is what it *means*. The coded columns
have to be named by hand, which is what section C does.

## C. Handling the coded columns properly

Two steps, and the first one is not code.

**Step 1 — decide, and write the list down.** No rule can do this for you. Open the data
dictionary and sort the low-cardinality integers into: genuine counts, ordered ratings,
and codes. Only you can tell that `MS_SubClass` 20 and 60 are labels while `Fireplaces`
2 really is twice `Fireplaces` 1.

**Step 2 — cast them to text before routing them.** The categorical branch imputes with
the string `"Missing"`, and pandas will not put a string in an `int64` column:

```
ValueError: fill_value='Missing' (of type <class 'str'>) cannot be cast to the
input data that is dtype('int64').
```

Here are two helpers worth copying into your own work.

In [ ]:
def sort_columns(df, outcome, coded=(), drop=()):
    """Split columns into (numeric, categorical), honouring your judgment calls.

    coded : integer columns that are really categories. You name these, because
            no dtype rule can find them.
    drop  : identifiers and anything else that must not reach the model.
    """
    usable = df.drop(columns=[outcome, *drop], errors="ignore")
    categorical = [c for c in usable.columns
                   if not pd.api.types.is_numeric_dtype(usable[c]) or c in coded]
    numeric = [c for c in usable.columns if c not in categorical]
    return numeric, categorical


def cast_coded(df, coded):
    """Make coded columns text, so the categorical branch can impute them."""
    out = df.copy()
    for c in coded:
        out[c] = out[c].astype(str)
    return out


# The judgment calls, stated once and reused.
CODES = ["MS_SubClass", "Mo_Sold"]        # labels that happen to be integers
DROP = [c for c in ames.columns if ames[c].is_unique]   # Order, PID

num, cat = sort_columns(ames, outcome="SalePrice", coded=CODES, drop=DROP)
fixed = cast_coded(ames, CODES)

print(f"  dropped as identifiers : {DROP}")
print(f"  moved to categorical   : {CODES}")
print(f"  -> {len(num)} numeric, {len(cat)} categorical")
print(f"  MS_SubClass is now categorical? {'MS_SubClass' in cat}")


def score(num, cat, data=ames):
    Xtr, Xte, ytr, yte = train_test_split(
        data[num + cat], data["SalePrice"], test_size=0.25, random_state=764)
    steps = []
    if num:
        steps.append(("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                                       ("scale", StandardScaler())]), num))
    if cat:
        steps.append(("cat", Pipeline([("fill", SimpleImputer(strategy="constant",
                                                              fill_value="Missing")),
                                       ("encode", OneHotEncoder(handle_unknown="ignore"))]), cat))
    pipe = Pipeline([("prep", ColumnTransformer(steps)),
                     ("model", LinearRegression())]).fit(Xtr, ytr)
    return r2_score(yte, pipe.predict(Xte)), pipe.named_steps["prep"].transform(Xtr[:5]).shape[1]


no_ids = [c for c in all_numeric if c not in ids]

print(f"  {'columns used':<40}{'in':>5}{'encoded':>9}{'R2':>9}")
for label, num, cat, data in [
    ("all numeric",                     all_numeric, [],              ames),
    ("all categorical",                 [],          all_categorical, ames),
    ("everything, as the dtypes fall",  all_numeric, all_categorical, ames),
    ("everything, identifiers dropped", no_ids,      all_categorical, ames),
    ("...and coded columns as categories", num, cat, fixed),
]:
    r, n = score(num, cat, data)
    print(f"  {label:<40}{len(num) + len(cat):>5}{n:>9}{r:>9.3f}")

Two things to take from that table, and the second one matters more.

**All the categorical columns alone beat all the numeric ones** — 0.819 against 0.784.
If you spent Thursday picking numbers and ignoring the text columns, that is worth
knowing.

**Cleaning up the column assignment barely moves the score.** 0.848 → 0.849 → 0.850.
Dropping `PID` costs you essentially nothing, and treating `MS_SubClass` properly buys
essentially nothing.

⚠ **Do it anyway, and notice that "it scores the same" is the argument *for* it, not
against.** A model that keeps `PID` is relying on the Story County parcel numbering
scheme to stand in for location. It scores the same today and it is indefensible the
moment anyone asks why it works — and it will not survive a renumbering, a different
county, or a reviewer. *Smallest you can defend* was the brief.